### Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
A function or coroutine to execute.

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='<think>\nOkay, so I need to figure out why parrots talk. Let\'s start by recalling what I know about parrots. They\'re birds, right? And some species are known for mimicking human speech. But why do they do that in the first place? Maybe it\'s related to their natural behavior. I remember reading that parrots are social animals, so talking might be a way to communicate with their flock. But in the wild, do they actually talk to each other like we do? Probably not in words, but maybe they use vocalizations to interact.\n\nWait, maybe the reason they mimic human speech is because of their environment. If they\'re kept as pets, they might learn to talk to interact with their owners. But even wild parrots can mimic sounds. So there\'s a general ability in parrots to copy sounds. What\'s the biological reason for that? I think it has to do with their brain structure. They have a part called the "song system" which helps with vocal learning. Maybe similar to how humans hav

In [2]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])

In [3]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Boston. I need to use the get_weather function. The function requires a location parameter. Boston is the location here. So I should call get_weather with location set to "Boston". Let me make sure there\'s no typo. Everything looks good. Let me format the tool call correctly.\n', 'tool_calls': [{'id': 'hp3pr0a3m', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 92, 'prompt_tokens': 154, 'total_tokens': 246, 'completion_time': 0.150344011, 'completion_tokens_details': {'reasoning_tokens': 68}, 'prompt_time': 0.006020131, 'prompt_tokens_details': None, 'queue_time': 0.371013327, 'total_time': 0.156364142}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_d58dbe76cd', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_ru

Tool Execution Loops

In [4]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

The weather in Boston is sunny.


In [5]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. I need to use the get_weather function. The function requires a location parameter. Boston is the location here. So I should call get_weather with location set to "Boston". Let me make sure there are no typos. Everything looks good. I\'ll format the tool call as specified.\n', 'tool_calls': [{'id': '51ppcrzqd', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 94, 'prompt_tokens': 153, 'total_tokens': 247, 'completion_time': 0.13955493, 'completion_tokens_details': {'reasoning_tokens': 70}, 'prompt_time': 0.006730778, 'prompt_tokens_details': None, 'queue_time': 0.054185901, 'total_time': 0.146285708}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish

User
  ↓
LLM
  ↓
Tool Call Request
  ↓
Python Function Executes
  ↓
Result Returned
  ↓
LLM Reads Result
  ↓
Final Natural Language Answer